# Real-time Communication

In [1]:
import networkx as nx 
from dataXplorer import JupyterClient
from dataXplorer import AnalysisToolkit, SessionManager, TextureGenerator, ProjectFileManager, VisualizerSyncer

In [ ]:
# run backend (server) using buildandrun powershell script
client = JupyterClient()

#client.disconnect()

✅ Connected to /main

✅ Connected to /main
✅ Connected to /main


In [3]:
# get current directory
import os 
current_directory = os.getcwd()
print("Current Directory:", current_directory)

Current Directory: c:\Users\chris\Desktop\GIT\DataDiVR_WebApp


### Data set  "Social byzantine 14th century" : "People and Locations" + "Localities Network"

In [4]:
# PEOPLE AND LOCATIONS.xml" 
# ------------------------------------------

import networkx as nx 
import xml.etree.ElementTree as ET

def create_people_and_locations_graph_from_file(file_path):
    """
    Parse a "People and locations" XML (same structure as "Only People.xml") into a NetworkX Graph.
    Nodes: all nodeclasses (e.g., Agent, Location) with their properties. Node label uses 'Node Title' if present.
    Edges: all link elements from networks/network sections. Each edge receives link attributes and a 'layer' attribute.

    Returns:
        networkx.Graph: simple graph (no parallel edges) built from the XML.
    """
    G = nx.Graph()
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
    except FileNotFoundError:
        print(f"Error: File not found: {file_path}")
        return nx.Graph()
    except ET.ParseError as e:
        print(f"Error parsing XML {file_path}: {e}")
        return nx.Graph()

    # Parse nodes from all nodeclass entries under nodes
    for nodeclass in root.findall('.//nodes/nodeclass'):
        ntype = nodeclass.get('type') or 'unknown'
        for node_elem in nodeclass.findall('node'):
            nid = node_elem.get('id')
            if not nid:
                continue
            props = {}
            label = nid
            for prop in node_elem.findall('property'):
                pid = prop.get('id')
                pval = prop.get('value')
                if pid and pval:
                    props[pid] = pval
                    if pid == 'Node Title':
                        label = pval
            props['type'] = ntype
            G.add_node(nid, label=label, **props)

    # Parse links from all networks (keep network id/name as edge layer)
    for network in root.findall('.//networks/network'):
        layer = network.get('id') or network.get('name') or 'network'
        for link in network.findall('link'):
            s = link.get('source')
            t = link.get('target')
            if not s or not t:
                continue
            # collect link attributes (exclude source/target)
            edge_attrs = {k: v for k, v in link.attrib.items() if k not in ('source', 'target')}
            edge_attrs['layer'] = layer
            # ensure nodes exist (create minimal node if missing)
            if s not in G:
                G.add_node(s, label=s, type='unknown')
            if t not in G:
                G.add_node(t, label=t, type='unknown')
            # add or update edge attributes (Graph will keep one edge per node-pair)
            if G.has_edge(s, t):
                # merge attributes if edge already exists: prefer existing, but update with any new keys
                existing = G[s][t]
                merged = dict(existing)
                merged.update(edge_attrs)
                G[s][t].update(merged)
            else:
                G.add_edge(s, t, **edge_attrs)

    # Summary
    from collections import Counter
    types = [data.get('type', 'unknown') for _, data in G.nodes(data=True)]
    type_counts = Counter(types)
    print("--- People & Locations Graph Summary ---")
    print(f"Total nodes: {G.number_of_nodes()} (by type: {dict(type_counts)})")
    print(f"Total edges: {G.number_of_edges()}")

    # Ensure we return a simple nx.Graph object
    if not isinstance(G, nx.Graph):
        H = nx.Graph()
        for n, data in G.nodes(data=True):
            H.add_node(n, **data)
        for u, v, data in G.edges(data=True):
            H.add_edge(u, v, **data)
        G = H

    return G

In [5]:
path_peopleandlocations = "temp-files/historynetworks/data/Social Network Multilayer Byzantine Elite 14th Century/People and locations.xml" 
people_location_graph = create_people_and_locations_graph_from_file(path_peopleandlocations)

--- People & Locations Graph Summary ---
Total nodes: 2735 (by type: {'Agent': 2399, 'Location': 336})
Total edges: 8095


In [6]:
# check for layers in links 
unique_layers = set()
for u, v, attrs in people_location_graph.edges(data=True):
    layer = attrs.get('layer', '')
    unique_layers.add(layer)
print("Unique link layers in people-location graph:", unique_layers)

Unique link layers in people-location graph: {'Kinship', 'Agent x Location', 'Allegiance', 'Marriage', 'Friendship and Support', 'Diplomacy', 'Conflict'}


In [7]:
# LOCALITIES NETWORK.xml"
# ------------------------------------------



from datetime import datetime
import math

# parse localities file and build location_positions (safe to rebuild)
path_locations = "temp-files/historynetworks/data/Social Network Multilayer Byzantine Elite 14th Century/Localities network total.xml"
tree_loc = ET.parse(path_locations)
root_loc = tree_loc.getroot()
location_positions = {}
for nodeclass in root_loc.findall('.//nodes/nodeclass'):
    if nodeclass.get('type') != 'Location':
        continue
    for node_elem in nodeclass.findall('node'):
        nid = node_elem.get('id')
        if not nid:
            continue
        lat = lon = None
        for prop in node_elem.findall('property'):
            pid = prop.get('id')
            pval = prop.get('value')
            if pid == 'Latitude':
                try:
                    lat = float(pval)
                except (TypeError, ValueError):
                    lat = None
            elif pid == 'Longitude':
                try:
                    lon = float(pval)
                except (TypeError, ValueError):
                    lon = None
        if lat is not None and lon is not None:
            # convert lon/lat (degrees) to ECEF cartesian coordinates (meters)
            R = 6371000.0  # mean Earth radius in meters
            lon_r = math.radians(lon)
            lat_r = math.radians(lat)
            x = R * math.cos(lat_r) * math.cos(lon_r)
            y = R * math.cos(lat_r) * math.sin(lon_r)
            z = R * math.sin(lat_r)
            location_positions[nid] = (x, y, z)

In [8]:
location_positions

{'Constantinople': (4205792.13214578, 2328990.143138585, 4180521.3137052613),
 'Moscow': (2840221.142856378, 2188581.2185688238, 5266203.291689319),
 'Alexandria': (4722190.027256838, 2718414.111372218, 3301633.999335549),
 'Lesbos': (4422293.6100138165, 2210177.15771002, 4018467.015961164),
 'Thessalonike': (4450816.766838681, 1885949.9314481427, 4150067.9469254976),
 'Gratianupolis/Thrakien': (4328766.251006201,
  2067807.734460262,
  4192325.7170043504),
 'Kastoria': (4513519.27657328, 1756622.5559712187, 4139089.505658771),
 'Trikala': (4561336.122895273, 1822172.9336381608, 4057528.7520715385),
 'Didymoteichon': (4280463.520607844, 2133538.527551829, 4208953.124019972),
 'Rome': (4631110.910975102, 1025279.2886254556, 4253381.608857009),
 'Arta': (4613040.153586794, 1769240.7412042134, 4022348.6598079856),
 'Kephallenia': (4685441.133437941, 1758031.2530065293, 3942791.967443892),
 'Ioannina': (4582254.481137169, 1745975.6398648517, 4067450.544886422),
 'Tarent': (4629354.44974142

In [9]:

# normalize ECEF coordinates to 0-1 range per axis
normalized_positions = {}
if location_positions:
    xs = [p[0] for p in location_positions.values()]
    ys = [p[1] for p in location_positions.values()]
    zs = [p[2] for p in location_positions.values()]
    minx, maxx = min(xs), max(xs)
    miny, maxy = min(ys), max(ys)
    minz, maxz = min(zs), max(zs)
    dx = maxx - minx
    dy = maxy - miny
    dz = maxz - minz

    for nid, (x, y, z) in location_positions.items():
        nx = (x - minx) / dx if dx > 0 else 0.5
        ny = (y - miny) / dy if dy > 0 else 0.5
        nz = (z - minz) / dz if dz > 0 else 0.5
        normalized_positions[nid] = (nx, ny, nz)
    else:
        print("No location positions found to normalize.")

# assign positions with checks for existing pos and timestamp metadata
added = 0
skipped = 0
updated_with_no_timestamp = []

for node_id, ecef_pos in location_positions.items():
    if node_id not in people_location_graph:
        continue
    node_attrs = people_location_graph.nodes[node_id]
    # Use ECEF coordinates directly without normalization
    people_location_graph.nodes[node_id]['pos'] = ecef_pos
    print(f"ADD  {node_id}: pos set to {ecef_pos}")
    added += 1

print(f"Summary: added={added}, skipped={skipped}, preexisting_marked={len(updated_with_no_timestamp)}")

No location positions found to normalize.
ADD  Constantinople: pos set to (4205792.13214578, 2328990.143138585, 4180521.3137052613)
ADD  Moscow: pos set to (2840221.142856378, 2188581.2185688238, 5266203.291689319)
ADD  Alexandria: pos set to (4722190.027256838, 2718414.111372218, 3301633.999335549)
ADD  Lesbos: pos set to (4422293.6100138165, 2210177.15771002, 4018467.015961164)
ADD  Thessalonike: pos set to (4450816.766838681, 1885949.9314481427, 4150067.9469254976)
ADD  Gratianupolis/Thrakien: pos set to (4328766.251006201, 2067807.734460262, 4192325.7170043504)
ADD  Kastoria: pos set to (4513519.27657328, 1756622.5559712187, 4139089.505658771)
ADD  Trikala: pos set to (4561336.122895273, 1822172.9336381608, 4057528.7520715385)
ADD  Didymoteichon: pos set to (4280463.520607844, 2133538.527551829, 4208953.124019972)
ADD  Rome: pos set to (4631110.910975102, 1025279.2886254556, 4253381.608857009)
ADD  Arta: pos set to (4613040.153586794, 1769240.7412042134, 4022348.6598079856)
ADD  Ke

In [10]:
# retrieve location connections from "Localities network total.xml"
path_locations = "temp-files/historynetworks/data/Social Network Multilayer Byzantine Elite 14th Century/Localities network total.xml"
tree_loc = ET.parse(path_locations)
root_loc = tree_loc.getroot()
for link_elem in root_loc.findall('.//networks/network/link'):
    source_id = link_elem.get('source')
    target_id = link_elem.get('target')
    # Initialize link attributes dictionary
    link_attrs = {}
    # Get link attributes like weight if available
    for attr_name, attr_value in link_elem.attrib.items():
        if attr_name not in ['source', 'target']:
            link_attrs[attr_name] = attr_value
    link_attrs['layer'] = 'Location x Location'
    # Only add edge if both nodes exist in our graph
    if source_id and target_id and source_id in people_location_graph and target_id in people_location_graph:
        # Add the edge with any additional attributes
        people_location_graph.add_edge(source_id, target_id, **link_attrs)

print("--- People & Locations Graph Summary ---")
print(f"Total Nodes: {people_location_graph.number_of_nodes()}")
print(f"Total Edges: {people_location_graph.number_of_edges()}")


--- People & Locations Graph Summary ---
Total Nodes: 2735
Total Edges: 10311


In [11]:
# check all unique link attributes in graph 

unique_layers = set()
for u, v, attrs in people_location_graph.edges(data=True):
    layer = attrs.get('layer', '')
    unique_layers.add(layer)

print("Unique link layers in people-location graph:", unique_layers)


Unique link layers in people-location graph: {'Kinship', 'Agent x Location', 'Allegiance', 'Marriage', 'Location x Location', 'Friendship and Support', 'Diplomacy', 'Conflict'}


In [12]:
# check which nodes in people_location_graph have pos attribute
nodes_with_pos = [n for n, attrs in people_location_graph.nodes(data=True) if 'pos' in attrs]
print(f"Nodes with 'pos' attribute: {len(nodes_with_pos)} out of {  people_location_graph.number_of_nodes()} total nodes.") 


Nodes with 'pos' attribute: 308 out of 2735 total nodes.


In [13]:
import networkx as nx

In [14]:
people_location_graph.graph['projectname'] = "ByzNet_PxL"
people_location_graph.graph['info'] = "ByzNet-1400 visualises the multi-layered social, political, and kinship networks of the Late Byzantine elite " \
"(1282–1402 CE) using data derived from the PLP prosopography. The dataset connects 2402 individuals and 336 localities across ties of kinship, " \
"allegiance, friendship, conflict, diplomacy, and mobility, revealing how elite factions formed, fractured, " \
"and interacted during a period of civil war and imperial contraction. The project constructs a multiplex, temporal, " \
"and geospatial graph, enabling exploration of power clusters, conflict dynamics, family expansion, and the central role " \
"of Constantinople within a highly interconnected but increasingly polarised aristocratic world."

people_location_graph.graph["layoutname"] ='layout-3D'

nx.set_node_attributes(people_location_graph, {node: data.copy() for node, data in people_location_graph.nodes(data=True)}, name='annotation')
# delete all other attributes except 'annotation'
for node in people_location_graph.nodes():
    # Preserve 'pos' and 'annotation' keys
    pos = people_location_graph.nodes[node].get('pos', None)
    annotation = people_location_graph.nodes[node].get('annotation', {})
    
    # Clear all attributes
    people_location_graph.nodes[node].clear()
    
    # Restore 'pos' and 'annotation' keys
    if pos is not None:
        people_location_graph.nodes[node]['pos'] = pos
    people_location_graph.nodes[node]['annotation'] = annotation

In [15]:
# check node information 
for node, data in list(people_location_graph.nodes(data=True))[0:5]:
    print(f"Node ID: {node}, Data: {data}")

Node ID: Ααρὼν Αλέξιος, Data: {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Rank': 'Oikeios', 'Begin': '1393.0', 'End': '1393', 'PLP': '3.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent'}}
Node ID: Αβράμιος Ιωάννης, Data: {'annotation': {'label': 'Αβράμιος Ιωάννης', 'Function': 'Astrologe in Kpl, 1371 - 1390; Priester (?), vor 1371-05; Hs.-Schreiber, 1376 - 1381, Schriftsteller, Arzt', 'Begin': '1371.0', 'End': '1391.0', 'PLP': '57.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent'}}
Node ID: Αβράμιος Μανουήλ, Data: {'annotation': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Rank': 'Doulos', 'Begin': '1336.0', 'End': '1336', 'PLP': '58.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent'}}
Node ID: Αγάθων Μανουήλ, Data: {'annotation': {'label': 'Αγάθων Μανουήλ', 'Function': 'Senator in Kpl, 1409', 'Rank': 'Oikeios', 'Begin': '1402.0', 'End': '1402', 'PLP': '88.0', 'Secular/Ecclesias

In [16]:
# print node information if type is 'Location', limit to first 5
count = 0
for node, data in list(people_location_graph.nodes(data=True)):
    if data['annotation'].get('type') == 'Location':
        print(f"Node ID: {node}, Data: {data}")
        count += 1
        if count == 5:
            break

Node ID: Trebizond, Data: {'pos': (3697681.602159104, 3073510.105964675, 4179752.073698522), 'annotation': {'label': 'Trebizond', 'type': 'Location', 'Latitude': '41', 'Longitude': '39.733333', 'pos': (3697681.602159104, 3073510.105964675, 4179752.073698522)}}
Node ID: Venice, Data: {'pos': (4367234.131769609, 955072.5503892349, 4539244.811838149), 'annotation': {'label': 'Venice', 'type': 'Location', 'Latitude': '45.4375', 'Longitude': '12.335833', 'pos': (4367234.131769609, 955072.5503892349, 4539244.811838149)}}
Node ID: Ragusa, Data: {'pos': (4454467.107687815, 1456717.8237097922, 4315708.165366787), 'annotation': {'label': 'Ragusa', 'type': 'Location', 'Latitude': '36.921667', 'Longitude': '14.719444', 'pos': (4454467.107687815, 1456717.8237097922, 4315708.165366787)}}
Node ID: Constantinople, Data: {'pos': (4205792.13214578, 2328990.143138585, 4180521.3137052613), 'annotation': {'label': 'Constantinople', 'Latitude': '41.009167', 'Longitude': '28.975833', 'type': 'Location', 'pos

In [17]:
import romanize3

mapping = {}
# create mapping from node id to name
for node, data in people_location_graph.nodes(data=True):
    name = data['annotation'].get('name', '')
    mapping[node] = name

# add roman letters name as attribute 
for node, data in people_location_graph.nodes(data=True):
    name = node
    print(f"Original Name: {name}")

    roman_name = romanize3.__dict__['grc'].convert(name)
    print(f"Node: {node}, Transliterated: {roman_name}")

    data['annotation']['transliterated'] = roman_name

Original Name: Ααρὼν Αλέξιος
Node: Ααρὼν Αλέξιος, Transliterated: Aarὼn Alέcios
Original Name: Αβράμιος Ιωάννης
Node: Αβράμιος Ιωάννης, Transliterated: Abrάmios Iôάnnês
Original Name: Αβράμιος Μανουήλ
Node: Αβράμιος Μανουήλ, Transliterated: Abrάmios Manouήl
Original Name: Αγάθων Μανουήλ
Node: Αγάθων Μανουήλ, Transliterated: Agάhôn Manouήl
Original Name: Αγάλλων Μανουήλ
Node: Αγάλλων Μανουήλ, Transliterated: Agάllôn Manouήl
Original Name: Αγγελος Γεώργιος 183
Node: Αγγελος Γεώργιος 183, Transliterated: Aggelos Geώrgios 183
Original Name: Αγγελος Δημήτριος 190
Node: Αγγελος Δημήτριος 190, Transliterated: Aggelos Dêmήtrios 190
Original Name: Αγγελος Ιωάννης 202
Node: Αγγελος Ιωάννης 202, Transliterated: Aggelos Iôάnnês 202
Original Name: Αδριανός, Πέτρος Δούκας
Node: Αδριανός, Πέτρος Δούκας, Transliterated: Adrianόs, Pέtros Doύkas
Original Name: Αγάλος
Node: Αγάλος, Transliterated: Agάlos
Original Name: Αγγελίτζης
Node: Αγγελίτζης, Transliterated: Aggelίtzês
Original Name: Αγγελος
Node: Α

In [19]:
import networkx as nx 

# extract fixed positions
pos_fixed = {node: data['pos'] for node, data in people_location_graph.nodes(data=True) if 'pos' in data}

import cartoGRAPHs as cg
pos_3D_people = cg.layout_global_umap(people_location_graph, dim=3, n_neighbors=10, min_dist=0.1)
pos_3D_people_ = {key: list(value) for key, value in pos_3D_people.items()}

# pos_spring = nx.spring_layout(
#     people_location_graph,
#     dim=3,
#     #pos=pos_fixed,
#     #fixed=pos_fixed.keys(),
#     iterations=50,
#     seed=42,
# )

nx.set_node_attributes(people_location_graph, {node: pos for node, pos in pos_3D_people_.items()}, name='pos')

c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEBUG: in init: import done


c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [20]:
# color code for links: 
import networkx as nx 

# link colors based on type 
linkcolor = []
for u, v, attrs in people_location_graph.edges(data=True):
    layer = attrs.get('layer', '')
    if layer == 'Location x Location':
        linkcolor.append((255,0,0,200))  
    elif layer == 'Agent x Location ':
        linkcolor.append( (0,0,255,200))  
    else:
        linkcolor.append((0,0,255,200)) 

nx.set_edge_attributes(people_location_graph, { (u, v): color for (u, v), color in zip(people_location_graph.edges(), linkcolor)}, name='linkcolor')

# node colors based on type
node_colors = []
for node, data in people_location_graph.nodes(data=True):
    ntype = data.get('annotation', {}).get('type', '')
    if ntype == 'Agent':
        node_colors.append((0, 0, 255, 110))  
    elif ntype == 'Location':
        node_colors.append((255, 0, 0, 110))  
    else:
        node_colors.append((60,60,60,60))

nx.set_node_attributes(people_location_graph, {node: color for node, color in zip(people_location_graph.nodes(), node_colors)}, name='nodecolor')

In [21]:
# check node information 
for node, data in list(people_location_graph.nodes(data=True))[:5]:
    print(f"Node ID: {node}, Data: {data}")

Node ID: Ααρὼν Αλέξιος, Data: {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Rank': 'Oikeios', 'Begin': '1393.0', 'End': '1393', 'PLP': '3.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent', 'transliterated': 'Aarὼn Alέcios'}, 'pos': [0.2392353045, 0.3025403821, 0.3805675116], 'nodecolor': (0, 0, 255, 110)}
Node ID: Αβράμιος Ιωάννης, Data: {'annotation': {'label': 'Αβράμιος Ιωάννης', 'Function': 'Astrologe in Kpl, 1371 - 1390; Priester (?), vor 1371-05; Hs.-Schreiber, 1376 - 1381, Schriftsteller, Arzt', 'Begin': '1371.0', 'End': '1391.0', 'PLP': '57.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent', 'transliterated': 'Abrάmios Iôάnnês'}, 'pos': [0.4104597945, 0.1851900446, 0.6211168138], 'nodecolor': (0, 0, 255, 110)}
Node ID: Αβράμιος Μανουήλ, Data: {'annotation': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Rank': 'Doulos', 'Begin': '1336.0', 'End': '1336', 'PLP': '58.0', 'Secular/E

In [22]:
# Extract 'label' attribute for all nodes and set it as a new attribute 'name'
labels = {node: data.get('annotation', {}).get('label', 'Unknown') for node, data in people_location_graph.nodes(data=True)}
nx.set_node_attributes(people_location_graph, labels, name='name')

In [23]:
# check node information 
for node, data in list(people_location_graph.nodes(data=True))[:5]:
    print(f"Node ID: {node}, Data: {data}")

Node ID: Ααρὼν Αλέξιος, Data: {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Rank': 'Oikeios', 'Begin': '1393.0', 'End': '1393', 'PLP': '3.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent', 'transliterated': 'Aarὼn Alέcios'}, 'pos': [0.2392353045, 0.3025403821, 0.3805675116], 'nodecolor': (0, 0, 255, 110), 'name': 'Ααρὼν Αλέξιος'}
Node ID: Αβράμιος Ιωάννης, Data: {'annotation': {'label': 'Αβράμιος Ιωάννης', 'Function': 'Astrologe in Kpl, 1371 - 1390; Priester (?), vor 1371-05; Hs.-Schreiber, 1376 - 1381, Schriftsteller, Arzt', 'Begin': '1371.0', 'End': '1391.0', 'PLP': '57.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent', 'transliterated': 'Abrάmios Iôάnnês'}, 'pos': [0.4104597945, 0.1851900446, 0.6211168138], 'nodecolor': (0, 0, 255, 110), 'name': 'Αβράμιος Ιωάννης'}
Node ID: Αβράμιος Μανουήλ, Data: {'annotation': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Rank': 'Doulos', 'Begin

In [24]:
# print Graph info 
import networkx as nx 
print(f"Graph Name: {people_location_graph.graph.get('projectname', 'Unknown')}")
print(f"Graph Info: {people_location_graph.graph.get('info', 'No additional information available.')}")

# print graph nodes and attributes
for node, data in list(people_location_graph.nodes(data=True))[:5]:
    print(f"Node ID: {node}, Data: {data}")

# print graph node and edge numbers
print(f"Total Nodes: {people_location_graph.number_of_nodes()}")
print(f"Total Edges: {people_location_graph.number_of_edges()}")


Graph Name: ByzNet_PxL
Graph Info: ByzNet-1400 visualises the multi-layered social, political, and kinship networks of the Late Byzantine elite (1282–1402 CE) using data derived from the PLP prosopography. The dataset connects 2402 individuals and 336 localities across ties of kinship, allegiance, friendship, conflict, diplomacy, and mobility, revealing how elite factions formed, fractured, and interacted during a period of civil war and imperial contraction. The project constructs a multiplex, temporal, and geospatial graph, enabling exploration of power clusters, conflict dynamics, family expansion, and the central role of Constantinople within a highly interconnected but increasingly polarised aristocratic world.
Node ID: Ααρὼν Αλέξιος, Data: {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Rank': 'Oikeios', 'Begin': '1393.0', 'End': '1393', 'PLP': '3.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent', 'transliterated': 'Aarὼn Alέcios'}, 'pos': [0.2392353045, 0.

In [25]:
import networkx as nx 

# remap node ids to be numbers
people_location_graph = nx.relabel_nodes(people_location_graph, {old_id: new_id for new_id, old_id in enumerate(people_location_graph.nodes())})

print(list(people_location_graph.nodes(data=True))[:10])

[(0, {'annotation': {'label': 'Ααρὼν Αλέξιος', 'Function': 'Gesandter', 'Rank': 'Oikeios', 'Begin': '1393.0', 'End': '1393', 'PLP': '3.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent', 'transliterated': 'Aarὼn Alέcios'}, 'pos': [0.2392353045, 0.3025403821, 0.3805675116], 'nodecolor': (0, 0, 255, 110), 'name': 'Ααρὼν Αλέξιος'}), (1, {'annotation': {'label': 'Αβράμιος Ιωάννης', 'Function': 'Astrologe in Kpl, 1371 - 1390; Priester (?), vor 1371-05; Hs.-Schreiber, 1376 - 1381, Schriftsteller, Arzt', 'Begin': '1371.0', 'End': '1391.0', 'PLP': '57.0', 'Secular/Ecclesiastical': 'Sec', 'type': 'Agent', 'transliterated': 'Abrάmios Iôάnnês'}, 'pos': [0.4104597945, 0.1851900446, 0.6211168138], 'nodecolor': (0, 0, 255, 110), 'name': 'Αβράμιος Ιωάννης'}), (2, {'annotation': {'label': 'Αβράμιος Μανουήλ', 'Function': 'δοῦλος d. Παλαιολόγος Ανδρόνικος ΙΙΙ., gegen den er sich verschworen hatte.', 'Rank': 'Doulos', 'Begin': '1336.0', 'End': '1336', 'PLP': '58.0', 'Secular/Ecclesiastical': 'Sec', 'ty

In [26]:
import nx2json as nx2j 
nx2j.create_project(people_location_graph)

Successfully created the directory static/projects/ByzNet_PxL 
PROGRESS: loaded graph JSON...
PROGRESS: stored graph data...
PROGRESS: stored layouts...
PROGRESS: stored node info...
PROGRESS: made node position textures...
PROGRESS: made textures for node colors...
PROGRESS: made textures for links...
PROGRESS: made textures for linkcolors...
PROGRESS: writing json files for project and nodes...
Project created successfully.


## realtime manipulation 

In [27]:
# see if new project is in projectlist
import GlobalData as GD

allprojects_updated = []
for i, proj in enumerate(GD.listProjects()):
    allprojects_updated.append((i,proj))
allprojects_updated

[(0, 'AE_Memes_2022'),
 (1, 'ARS23_memes'),
 (2, 'ByzNet-1400-people-and-locations'),
 (3, 'ByzNet-1400-people-only'),
 (4, 'ByzNet_PxL'),
 (5, 'CDK5'),
 (6, 'CircLadderGraph-xsmall'),
 (7, 'diffusion'),
 (8, 'JSON_autocore'),
 (9, 'JSON_barbellgraph'),
 (10, 'JSON_Zachary'),
 (11, 'Microplastics_HumanHealth'),
 (12, 'Pesticides_HumanHealth'),
 (13, 'PG_NEW'),
 (14, 'Powergrid_Europe'),
 (15, 'PPI_brain_infarction'),
 (16, 'PPI_joel_daniel_aryan'),
 (17, 'PPI_joel_daniel_aryan_C'),
 (18, 'PPI_joel_daniel_aryan_test_1'),
 (19, 'PPI_networkcartoGRAPHs'),
 (20, 'Realtime-project'),
 (21, 'Sphere_Torus'),
 (22, 'Teapot'),
 (23, 'Template'),
 (24, 'test'),
 (25, 'TheMandelbulb_edges'),
 (26, 'XX')]

In [ ]:
# select a project to work with
sel_id = 2 #4
sel_name = allprojects_updated[sel_id][1]

# load the project data
session = SessionManager(sel_id, sel_name, client)
session.load_graph_from_project()
session.reload_project()

# initialize 
file_mgr = ProjectFileManager(session)
tex_gen = TextureGenerator(session)
syncer = VisualizerSyncer(session)

Session Graph loaded from project folder. 
Project name:  ByzNet_PxL
Data: Nodes: 2735 Links: 10311


In [32]:
G = session.load_graph_from_project()

# get nodenames from node attributes
nodenames_session = [data.get("name") for _, data in G.nodes(data=True)]
nodenames_session = [name for name in nodenames_session if name is not None]

# rename nodes back to original names
mapping = dict(zip(G.nodes(), nodenames_session))
G = nx.relabel_nodes(G, mapping)
G.nodes()

Session Graph loaded from project folder. 
Project name:  ByzNet_PxL
Data: Nodes: 2735 Links: 10311


NodeView(('Ααρὼν Αλέξιος', 'Αβράμιος Ιωάννης', 'Αβράμιος Μανουήλ', 'Αγάθων Μανουήλ', 'Αγάλλων Μανουήλ', 'Αγγελος Γεώργιος 183', 'Αγγελος Δημήτριος 190', 'Αγγελος Ιωάννης 202', 'Αδριανός, Πέτρος Δούκας', 'Αγάλος', 'Αγγελίτζης', 'Αγγελος', 'Αγγελος, Θεόδωρος Κομνηνός', 'Αγγελος Μανουήλ', 'Αγγελος, Μιχαὴλ Δούκας', '<Αγγελος>, Νικηφόρος ΙΙ. Δούκας', 'Αθανάσιος 422', 'Αθιανός', 'Ακροπολίτης Λέων', 'Ακροπολίτης Μανουήλ', 'Ακροπολίτης Μελχισεδέκ', 'Ακτουάριος (?) Νικόλαος', 'Αλέξιος, Statthalter', 'Αλέξιος, Dux', 'Αλησέρης (Mehmed Bey Germiyanoglu)', 'Αλληλούϊας', 'Αλουσιάνος', 'Αλουσιάνος Θωμᾶς <Δούκας>', 'Αλουψοῦ Αντώνιος', 'Αμπαρ Ιωάννης', 'Ανδρέας 928', 'Ανδρονικόπουλος Ιωάννης', 'Ανδρόνικος 952', 'Ανδρόνικος 959', 'Αντίοχος 1038', 'Απελμενέ 1151', 'Απελμενέ 1152', 'Απλησφάρης Ιωάννης', 'Απόκαυκος, Γεώργιος Δούκας', 'Απόκαυκος Ιωάννης, Megas Primikerios', 'Απόκαυκος, Μανουὴλ Δούκας Δισύπατος', 'Αρῆγος (Enrico)', 'Αρμενόπουλος Δημήτριος', 'Αρμπενος', 'Αρτῶτος 1447', 'Αρχοντίτζης', 'Αρχοντί

In [33]:
# initialize analysis toolkit
tools = AnalysisToolkit(session, tex_gen, syncer, file_mgr)

In [ ]:
# # make importance layout 
# import cartoGRAPHs as cg

# # transform multi graph to simple graph for layout
# simple_graph = nx.Graph()
# simple_graph.add_nodes_from(G.nodes(data=True))
# simple_graph.add_edges_from(G.edges(data=True))

# posG_3D_global = cg.layout_global_umap(simple_graph, dim=3,n_neighbors=10, min_dist=0.1)

# # replace nodes from geolocations in dict 
# posG_all = {}
# for node in G.nodes():
#     if node in pos_fixed:
#         posG_all[node] = pos_fixed[node]
#     else:
#         posG_all[node] =posG_3D_global[node]
    

# # make node texture for importance layout
# tex_gen.generate_node_position_texture(
#     posG_all,
#     texture_name="z_GlobalLayout_3D",
#     save=True
# )


c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


✅ Connected to /main
✅ Connected to /main


('static/projects/ByzNet_PxL/layouts/z_GlobalLayout_3D.bmp',
 'static/projects/ByzNet_PxL/layoutsl/z_GlobalLayout_3Dl.bmp')

In [36]:
unique_layers

{'Agent x Location',
 'Allegiance',
 'Conflict',
 'Diplomacy',
 'Friendship and Support',
 'Kinship',
 'Location x Location',
 'Marriage'}

In [37]:
# layout : 
# location links in black
# types of links by color

# assign colors to links based on unique_layers

layer_color = {
    'Location x Location': (100,100,100, 200),  
    'Agent x Location': (200, 200, 200, 200), 
    'Allegiance': (0 ,0,255, 200),  
    'Conflict': (0,191,255, 200),  
    'Diplomacy': (43,255, 0, 200),   
    'Friendship and Support': (255, 165, 0, 200), 
    'Kinship': (255, 100, 0, 200), 
    'Marriage': (255, 0, 0, 200)
}

# Use the predefined layer_color dictionary for efficiency
linkcolor = [
    layer_color.get(attrs.get('layer', ''), (255, 0, 0, 200))  # Default to red if layer not in layer_color
    for _, _, attrs in people_location_graph.edges(data=True)
]

nx.set_edge_attributes(people_location_graph, { (u, v): color for (u, v), color in zip(people_location_graph.edges(), linkcolor)}, name='linkcolor')

In [38]:
# print edge attributes for first 5 edges
for u, v, attrs in list(people_location_graph.edges(data=True))[:5]:
    print(f"Edge ({u}, {v}): Attributes: {attrs}")

Edge (0, 2403): Attributes: {'layer': 'Agent x Location', 'linkcolor': (200, 200, 200, 200)}
Edge (0, 2402): Attributes: {'layer': 'Agent x Location', 'linkcolor': (200, 200, 200, 200)}
Edge (0, 721): Attributes: {'layer': 'Allegiance', 'linkcolor': (0, 0, 255, 200)}
Edge (0, 566): Attributes: {'layer': 'Friendship and Support', 'linkcolor': (255, 165, 0, 200)}
Edge (1, 2405): Attributes: {'layer': 'Agent x Location', 'linkcolor': (200, 200, 200, 200)}


In [39]:
# make one texture per link type
# where links of that type are colored based on dictionary, others are dark grey transparent
for layer in unique_layers:
    print(f"Generating texture for layer: {layer}")

    layer_edges = {
        (u, v): layer_color[layer] if attrs.get('layer', '') == layer else (0,0,0,0)
        for u, v, attrs in people_location_graph.edges(data=True)
    }

    tex_gen.generate_link_color_texture(
        layer_edges,
        texture_name=f"z_Layer_{layer.replace(' ', '_')}",
        save=True       
    )
    
    print(f"Texture for layer {layer} generated successfully.")


Generating texture for layer: Kinship
Texture for layer Kinship generated successfully.
Generating texture for layer: Agent x Location
Texture for layer Agent x Location generated successfully.
Generating texture for layer: Allegiance
Texture for layer Allegiance generated successfully.
Generating texture for layer: Marriage
Texture for layer Marriage generated successfully.
Generating texture for layer: Location x Location
Texture for layer Location x Location generated successfully.
Generating texture for layer: Friendship and Support
Texture for layer Friendship and Support generated successfully.
Generating texture for layer: Diplomacy
Texture for layer Diplomacy generated successfully.
Generating texture for layer: Conflict
Texture for layer Conflict generated successfully.


✅ Connected to /main


In [40]:
# make node rgb textures 
# where all nodes are colored if they appear in links of that type

for layer in unique_layers:
    print(f"Generating node texture for layer: {layer}")

    # find all nodes that appear in edges of this layer
    nodes_in_layer = set()
    for u, v, attrs in people_location_graph.edges(data=True):
        if attrs.get('layer', '') == layer:
            nodes_in_layer.add(u)
            nodes_in_layer.add(v)

    layer_node_colors = {
        node: layer_color[layer] if node in nodes_in_layer else (60,60,60,60)
        for node in people_location_graph.nodes()
    }

    tex_gen.generate_node_color_texture(
        layer_node_colors,
        texture_name=f"z_NodeColor_Layer_{layer.replace(' ', '_')}",
        save=True       
    )
    
    print(f"Node texture for layer {layer} generated successfully.")

Generating node texture for layer: Kinship
✅ Connected to /main
Node texture for layer Kinship generated successfully.
Generating node texture for layer: Agent x Location
Node texture for layer Agent x Location generated successfully.
Generating node texture for layer: Allegiance
Node texture for layer Allegiance generated successfully.
Generating node texture for layer: Marriage
Node texture for layer Marriage generated successfully.
Generating node texture for layer: Location x Location
Node texture for layer Location x Location generated successfully.
Generating node texture for layer: Friendship and Support
Node texture for layer Friendship and Support generated successfully.
Generating node texture for layer: Diplomacy
Node texture for layer Diplomacy generated successfully.
Generating node texture for layer: Conflict
Node texture for layer Conflict generated successfully.


✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main


In [ ]:
import numpy as np

# create layouts for each layer - with grey nodes in periphery (sphere) 
# use cartographs functional layout - 
# start with sphere with all nodes 
# move them into center based on their connections in that layer


# spherical positions for all nodes
pos_pre_all = {}
radius = 1 # radius for peripheral nodes

total_nodes = len(people_location_graph.nodes())

angle_step = 360 / total_nodes  # ensure unique positions by dividing the circle evenly
current_angle = 0

# evenly distribute nodes on a spherical mesh
phi = np.linspace(0, np.pi, total_nodes)  # polar angle
theta = np.linspace(0, 2 * np.pi, total_nodes, endpoint=False)  # azimuthal angle

for i, node in enumerate(people_location_graph.nodes()):
    # calculate spherical coordinates
    x = radius * np.sin(phi[i]) * np.cos(theta[i])
    y = radius * np.sin(phi[i]) * np.sin(theta[i])
    z = radius * np.cos(phi[i])
    pos_pre_all[node] = (x, y, z)

# generate layout per layer
for layer in unique_layers:
    print(f"Generating layout for layer: {layer}")

    # find all nodes that appear in edges of this layer
    nodes_in_layer = set()
    for u, v, attrs in people_location_graph.edges(data=True):
        if attrs.get('layer', '') == layer:
            nodes_in_layer.add(u)
            nodes_in_layer.add(v)

    # create subgraph for this layer
    subgraph = people_location_graph.subgraph(nodes_in_layer).copy()
    print(f" Subgraph for layer {layer} has {subgraph.number_of_nodes()} nodes and {subgraph.number_of_edges()} edges.")

    # compute layout
    pos_layer = cg.layout_global_umap(subgraph, dim=3, n_neighbors=20, min_dist=0.1)

    # expand to all nodes, placing non-layer nodes on sphere periphery
    pos_all = {}
    for node in people_location_graph.nodes():
        if node in pos_layer:
            pos_all[node] = pos_layer[node]
        else:
            pos_all[node] = pos_pre_all[node]

    # generate texture
    tex_gen.generate_node_position_texture(
        pos_all,
        texture_name=f"z_Layout_3D_Layer_{layer.replace(' ', '_')}",
        save=True       
    )
    
    print(f"Layout texture for layer {layer} generated successfully.")

Generating layout for layer: Kinship
 Subgraph for layer Kinship has 1268 nodes and 2903 edges.


c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


Layout texture for layer Kinship generated successfully.
Generating layout for layer: Agent x Location
 Subgraph for layer Agent x Location has 1781 nodes and 8830 edges.


c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


Layout texture for layer Agent x Location generated successfully.
Generating layout for layer: Allegiance
 Subgraph for layer Allegiance has 1004 nodes and 2463 edges.


...

c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


Layout texture for layer Diplomacy generated successfully.
Generating layout for layer: Conflict
 Subgraph for layer Conflict has 433 nodes and 1236 edges.


c:\Users\chris\Desktop\GIT\DataDiVR_WebApp\venv\lib\site-packages\umap\umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


Layout texture for layer Conflict generated successfully.


✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
✅ Connected to /main
